# 第2回：LLMを動かして理解する（後半）

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session02/session02_llm_basics.ipynb)

このノートブックでは LLMの動作を手を動かしながら体験する。


## 準備
---

> **GPU ランタイムに切り替えて使用する（T4 以上推奨）。**  
> Colab メニュー → ランタイム → ランタイムのタイプを変更 → T4 GPU  
> GPUランタイムが利用できない場合はCPUでも可（ただし遅い）

### パッケージのインストール

In [ ]:
# パッケージのインストール
!pip install -q openai
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!uv pip install "vllm==0.19.1" --torch-backend=cu129 -q
#!uv pip install --system "vllm==0.19.1" -q

---
## A. OpenAI API（クローズドモデル）

モデルの重みは公開されず、`OPENAI_API_KEY` を付けて OpenAI のサーバーにリクエストを送り、結果を受け取る方式。

**2系統のエンドポイント**

| エンドポイント | SDK メソッド | 入力形式 | 主な用途 |
|---|---|---|---|
| `POST /v1/completions` | `client.completions.create` | `prompt`（単一の文字列） | プロンプトの**続き**を生成。非チャットモデルで使用 |
| `POST /v1/chat/completions` | `client.chat.completions.create` | `messages`（`role` + `content` の配列） | `system` / `user` / `assistant` の**対話形式**。現在の主流 |


> - 現在の OpenAI は、ツール呼び出しや状態管理を統合した上位の **Responses API**（`/v1/responses`）も提供しているが、本ノートブックでは扱わない。
> - Anthropic・Google は**チャット形式に一本化**されており、純粋な「プロンプトの続きを書く」用途の API は提供していない。
> - OpenAI も `/v1/completions` で使えるモデル（`gpt-3.5-turbo-instruct`・`babbage-002`・`davinci-002`）を **2026年9月28日にすべて廃止予定**（[Deprecations](https://developers.openai.com/api/docs/deprecations)）。後継の completions 対応モデルは用意されておらず、**Completions API は事実上の廃止**となる。


`/v1/completions` と `/v1/chat/completions` という仕様は、**業界の事実上の準標準**になっている。各種の LLM 実行基盤（ランタイム／推論サーバー）は、自前のモデルを **OpenAI 互換エンドポイント**として公開しており、`openai` クライアントの `base_url` を差し替えるだけで、コードをほぼ変えずにモデルを切り替えられる。また、**Anthropic・Google などのクラウド**も OpenAI 互換エンドポイントを用意している。

### 主なリクエストパラメータ

生成の挙動は次のパラメータで制御する（出力のばらつきと長さに関わる重要なものを抜粋）。

- **`model`**：使用するモデル名（例：`gpt-4o-mini`）。
- **`messages` / `prompt`**：入力本体。`chat.completions` は `messages`、`completions` は `prompt`。
- **`temperature`**：出力のランダム性（0〜2 程度）。`0` でほぼ決定的、値を上げるほど多様になる。
- **`max_tokens`**：生成する最大トークン数。応答の長さ・コスト・打ち切りに直結。
- **`top_p`**：確率上位 p% の候補だけからサンプリング（核サンプリング）。`temperature` と並ぶ多様性の制御で、通常はどちらか一方を調整。
- **`stop`**：この文字列が出たら生成を止める区切り指定。
- **`stream`**：`True` でトークンを逐次ストリーミング受信。

### `OPENAI_API_KEY` の設定

OpenAI API キー（`sk-...`）を [OpenAI Platform](https://platform.openai.com/api-keys) で取得し、次の方法で設定


1. Colab 左サイドバーの **鍵アイコン（シークレット）** を開く
2. **「新しいシークレットを追加」** をクリック
3. 名前に `OPENAI_API_KEY`、値に `sk-...` を入力して保存
4. **「ノートブックアクセス」** をオンにする
5. 以下のセルを実行


In [ ]:
# OPENAI_API_KEYの設定
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

- `completions.create` (`gpt-3.5-turbo-instruct`) — 非チャットモデル

In [ ]:
from openai import OpenAI
openai_client = OpenAI() # `api_key`を指定しなければOPENAI_API_KEYが使用される

response = openai_client.completions.create(
    model='gpt-3.5-turbo-instruct',
    prompt='東京は日本の首都であり、',
    max_tokens=60,
    temperature=0.0,
)
print('[OpenAI Completions]')
print(response.model_dump_json(indent=2))

- `chat.completions.create` (`gpt-4o-mini`) — チャットモデル

In [ ]:
# チャットモデル：Chat Completions エンドポイント（system / user / assistant 形式）
response = openai_client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': '簡潔に答えてください。'},
        {'role': 'user',   'content': '東京について教えてください。'},
    ],
    max_tokens=100,
)
print('[OpenAI Chat Completions / チャットモデル]')
print(response.model_dump_json(indent=2))

- `chat.completions.create` の tool call — モデルに関数呼び出しを選ばせる

    `tools` に利用可能な関数の仕様を渡すと、モデルは必要に応じて「この関数をこの引数で呼んでほしい」という構造化データを返す。実際に関数を実行するのは Python 側の役割で、その結果を `role='tool'` のメッセージとして会話に戻すと、モデルが最終回答を生成する。

In [ ]:
# Tool calling：モデルが必要な関数と引数を選び、Python側で実行する
import json


def get_weather(location: str, unit: str = 'celsius') -> dict:
    """デモ用の固定データ。実運用ではここで外部APIやDBを呼び出す。"""
    weather_by_city = {
        '東京': {'temperature': 23, 'condition': '晴れ'},
        '大阪': {'temperature': 25, 'condition': 'くもり'},
    }
    weather = weather_by_city.get(location, {'temperature': 20, 'condition': '不明'})
    return {'location': location, 'unit': unit, **weather}


tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': '指定した都市の現在の天気を取得する',
            'parameters': {
                'type': 'object',
                'properties': {
                    'location': {
                        'type': 'string',
                        'description': '都市名。例: 東京、大阪',
                    },
                    'unit': {
                        'type': 'string',
                        'enum': ['celsius', 'fahrenheit'],
                        'description': '温度の単位',
                    },
                },
                'required': ['location'],
            },
        },
    }
]

messages = [
    {'role': 'system', 'content': '必要ならツールを使い、簡潔に日本語で答えてください。'},
    {'role': 'user', 'content': '東京の現在の天気を確認して、服装のアドバイスをください。'},
]

first_response = openai_client.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    tools=tools,
)

assistant_message = first_response.choices[0].message
print('[OpenAI Tool Call / 1回目の応答]')
print(assistant_message.model_dump_json(indent=2))


In [ ]:

messages.append(assistant_message.model_dump(exclude_none=True))

available_functions = {
    'get_weather': get_weather,
}

for tool_call in assistant_message.tool_calls or []:
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    function_result = available_functions[function_name](**function_args)

    messages.append(
        {
            'role': 'tool',
            'tool_call_id': tool_call.id,
            'content': json.dumps(function_result, ensure_ascii=False),
        }
    )

final_response = openai_client.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
)

print('\n[OpenAI Tool Call / 最終回答]')
print(final_response.choices[0].message.content)

---
## B. vLLM

- GPU を使った高速 LLM 推論フレームワーク
- PagedAttention により大量リクエストを効率処理
- OpenAI 互換 API サーバーもサポート

> **GPU ランタイムが必要（T4 以上）**

### B-1. Python API

- ベースモデル（`Qwen/Qwen2.5-0.5B`）

In [ ]:
from vllm import LLM, SamplingParams

llm_base = LLM(model='Qwen/Qwen2.5-0.5B', max_model_len=512, dtype='float16', gpu_memory_utilization=0.8)

sampling_params = SamplingParams(temperature=0.5, max_tokens=60)
outputs = llm_base.generate(['東京は日本の首都であり、'], sampling_params)

print('[vLLM / ベースモデル / テキスト補完]')
print(outputs[0].outputs[0].text)

メモリ解放

In [ ]:
from vllm.distributed import cleanup_dist_env_and_memory
import torch
import gc

del llm_base
cleanup_dist_env_and_memory()

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

- チャットモデル（`Qwen/Qwen2.5-0.5B-Instruct`）

In [ ]:
llm_chat = LLM(model='Qwen/Qwen2.5-0.5B-Instruct', max_model_len=512, dtype='float16', gpu_memory_utilization=0.8)

conversation = [
    {'role': 'system', 'content': '簡潔に答えてください。'},
    {'role': 'user',   'content': 'AIエージェントとは何ですか？'},
]
sampling_params = SamplingParams(temperature=0.0, max_tokens=150)
outputs = llm_chat.chat(messages=[conversation], sampling_params=sampling_params)

print('[vLLM / チャットモデル / 指示への回答]')
print(outputs[0].outputs[0].text)

メモリ解放

In [ ]:
del llm_chat
cleanup_dist_env_and_memory()

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

### B-2. API サーバー（OpenAI 互換）

- ベースモデル（`Qwen/Qwen2.5-0.5B`）

  ターミナルを開き、次のコマンドでサーバーを起動する。

  ```bash
  python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-0.5B \
    --port 8000 \
    --max-model-len 512 \
    --dtype float16
  ```

  `Application startup complete` のようなログが出たら準備完了。下のセルからクライアントで接続します（サーバーは別ターミナルで起動したまま使い、終了時は `Ctrl+C` で停止）。

  > ベースモデルなので、クライアント側は `completions.create`（プロンプトの続きを生成）を使う。

In [ ]:
from openai import OpenAI
import json

vllm_client = OpenAI(base_url='http://localhost:8000/v1', api_key='dummy')

response = vllm_client.completions.create(
    model='Qwen/Qwen2.5-0.5B',
    prompt='東京は日本の首都であり、',
    max_tokens=150,
)
print('[vLLM OpenAI 互換サーバー / 非チャット]')
print(response.model_dump_json(indent=2))


- チャットモデル（`Qwen/Qwen2.5-0.5B-Instruct`）

  ターミナルを開き、次のコマンドでサーバーを起動する。

  ```bash
  python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --port 8080 \
    --max-model-len 512 \
    --dtype float16
  ```

  `Application startup complete` のようなログが出たら準備完了。下のセルからクライアントで接続する（サーバーは別ターミナルで起動したまま使い、終了時は `Ctrl+C` で停止）。

In [ ]:
from openai import OpenAI

vllm_client = OpenAI(base_url='http://localhost:8080/v1', api_key='dummy')

response = vllm_client.chat.completions.create(
    model='Qwen/Qwen2.5-0.5B-Instruct',
    messages=[
        {'role': 'system', 'content': '簡潔に答えてください。'},
        {'role': 'user',   'content': 'AIエージェントとは何ですか？'},
    ],
    max_tokens=150,
)
print('[vLLM OpenAI 互換サーバー / チャット]')
print(response.model_dump_json(indent=2))

---
## C. Ollama

バックエンドに `llama.cpp` を使用。  
**CLI**実行と **REST API サーバー**に対応

バックグラウンドで起動
```bash
nohup ollama serve > ollama.log 2>&1 &
```

### C-1. CLI実行

ターミナルを開き、`ollama run <model> "<prompt>"` を実行

```
ollama run qwen2.5:0.5b 'AIエージェントとは何ですか？簡潔に答えてください。' --nowordwrap
```

### C-2. API

| エンドポイント | 用途 |
|---------------|------|
| `POST /api/generate` | テキスト補完（`prompt` キー） |
| `POST /api/chat` | チャット補完（`messages` キー、OpenAI 互換） |

APIサーバーとして使用する場合は事前にモデルをダウンロードしておく

```bash
ollama pull qwen3:0.6b
```

In [ ]:
# サーバー疎通確認（ターミナルでセットアップ済みの前提）
import requests

response = requests.get('http://localhost:11434/api/tags')
print('[Ollama サーバー疎通OK / インストール済みモデル]')
for model in response.json().get('models', []):
    print(' -', model['name'])

- テキスト補完API

In [ ]:
import requests
import json

# /api/generate：テキスト補完
payload = {
    'model': 'qwen3:0.6b', # ※Ollamaのqwen3:0.6bはチャット形式のインストラクション・モデルであるがテキスト補完でも動く
    'prompt': '東京は日本の首都であり、',
    'stream': False,
    'think': False,  # qwen3 は thinking モデルのため、<think>...</think> の出力を抑止する
    'options': {'temperature': 0.0, 'num_predict': 60},
}
response = requests.post('http://localhost:11434/api/generate', json=payload)
print('[Ollama /api/generate / テキスト補完]')
print(json.dumps(response.json(), ensure_ascii=False, indent=2))

- チャット補完API

In [ ]:
# /api/chat：チャット補完（OpenAI 互換メッセージ形式）
payload = {
    'model': 'qwen3:0.6b',
    'messages': [
        {'role': 'system', 'content': '簡潔に答えてください。'},
        {'role': 'user',   'content': 'AIエージェントとは何ですか？'},
    ],
    'stream': False,
}
response = requests.post('http://localhost:11434/api/chat', json=payload)
print('[Ollama /api/chat / チャット補完]')
print(json.dumps(response.json(), ensure_ascii=False, indent=2))

### C-3. OpenAI 互換 API

| エンドポイント | 用途 |
|---------------|------|
| `POST /v1/completions` | テキスト補完（`prompt` キー） |
| `POST /v1/chat/completions` | チャット補完（`messages` キー） |

- テキスト補完API

In [ ]:
import json
from openai import OpenAI

ollama_openai_client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
response = ollama_openai_client.completions.create(
    model='qwen3:0.6b',
    prompt='東京は日本の首都であり、',
    temperature=0.0,
    max_tokens=80,
)

print('[Ollama OpenAI互換 /v1/completions]')
print(response.model_dump_json(indent=2))

- チャット補完API

In [ ]:
response = ollama_openai_client.chat.completions.create(
    model='qwen3:0.6b',
    messages=[
        {'role': 'system', 'content': '簡潔に答えてください。'},
        {'role': 'user', 'content': '東京の有名な観光地を2つ挙げてください。'},
    ],
    temperature=0.0,
    max_tokens=80,
)

print('[Ollama OpenAI互換 /v1/chat/completions]')
print(response.model_dump_json(indent=2))